<a href="https://colab.research.google.com/github/malihasaeed/langraphai/blob/main/Business_%26__Enterprise_AI__Strategy%E2%80%93Intent_Advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "pydantic<=2.12.3" --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.5
    Uninstalling pydantic-2.12.5:
      Successfully uninstalled pydantic-2.12.5


In [ ]:
!pip install -U langgraph langchain langchain-openai pydantic -q

In [ ]:
import os
from typing import TypedDict, List, Literal

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("LANGRAPHAI_API_KEY")

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2
)

In [ ]:
class AdvisorState(TypedDict):
    query: str
    intent: Literal["strategy", "implementation", "unknown"]
    department: str
    response_log: List[str]

In [ ]:
def classify_business_intent(state: AdvisorState):
    system_prompt = """
You are an internal AI Enablement Advisor router for a mid-sized company.
Your job is to classify internal AI-related queries for deterministic routing.
You do not answer the query. You only return a label.
"""

    task_prompt = f"""
Classify the following internal query into EXACTLY ONE category:

- strategy → leadership-level, business impact, ROI, risk, adoption decisions
- implementation → execution details, tooling, architecture, steps, deployment
- unknown → unclear, incomplete, or unrelated

Query:
\"\"\"{state['query']}\"\"\"

Rules:
- Respond with only one label: strategy / implementation / unknown
- No punctuation
- No explanations
"""

    intent = llm.invoke(system_prompt + task_prompt).content.strip().lower()

    # Safety: normalize unexpected outputs
    if intent not in ["strategy", "implementation", "unknown"]:
        intent = "unknown"

    return {
        "intent": intent,
        "response_log": state["response_log"] + [f"[Router] intent={intent}"]
    }

In [ ]:
def ai_strategy_node(state: AdvisorState):
    system_prompt = """
You are a senior AI strategy advisor for an SME.
You advise leadership on AI adoption decisions.
Focus on business value, feasibility, risk, trade-offs, and decision clarity.
Avoid hype and avoid deep implementation detail unless asked.
If something is unknown, say what needs to be clarified.
"""

    task_prompt = f"""
Answer the internal strategy question below.

Output format:
1) Executive takeaway (1-2 lines)
2) Business impact (bullets)
3) Risks & constraints (bullets)
4) Recommendation (clear next step)
5) 3 clarifying questions (only if needed)

Internal Query:
\"\"\"{state['query']}\"\"\"
"""

    response = llm.invoke(system_prompt + task_prompt)

    return {
        "response_log": state["response_log"] + [response.content]
    }

In [ ]:
def ai_implementation_node(state: AdvisorState):
    system_prompt = """
You are an AI implementation planner inside an enterprise environment.
Your job is to produce a realistic plan with phases, dependencies, and risks.
Do not invent company-specific details. If missing info, ask targeted questions.
"""

    task_prompt = f"""
Create a practical implementation plan for the request below.

Constraints:
- Assume SME resources (small team, limited infra)
- Prefer simple, testable milestones
- Include security/privacy notes when relevant

Output format:
A) Summary (2-3 lines)
B) Phase plan (Phase 0..3 with bullets)
C) Tech stack suggestion (bullets)
D) Risks + mitigations (bullets)
E) What you need from the company (checklist)

Internal Request:
\"\"\"{state['query']}\"\"\"
"""

    response = llm.invoke(system_prompt + task_prompt)

    return {
        "response_log": state["response_log"] + [response.content]
    }

In [ ]:
def route_by_business_intent(state: AdvisorState):
    if state["intent"] == "strategy":
        return "ai_strategy"
    elif state["intent"] == "implementation":
        return "ai_implementation"
    else:
        return END

In [ ]:
graph = StateGraph(AdvisorState)

# Add nodes
graph.add_node("intent_classifier", classify_business_intent)
graph.add_node("ai_strategy", ai_strategy_node)
graph.add_node("ai_implementation", ai_implementation_node)

# Entry point
graph.set_entry_point("intent_classifier")

# Conditional routing
graph.add_conditional_edges(
    "intent_classifier",
    route_by_business_intent
)

# End edges
graph.add_edge("ai_strategy", END)
graph.add_edge("ai_implementation", END)

In [ ]:
advisor = graph.compile()

In [ ]:
state = {
    "query": "Should we adopt AI for our customer support operations this year?",
    "intent": "unknown",
    "department": "operations",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=strategy

1) **Executive takeaway:** Adopting AI for customer support can enhance efficiency and customer satisfaction, but careful consideration of implementation challenges and risks is essential.

2) **Business impact:**
   - Improved response times and availability, leading to higher customer satisfaction.
   - Reduction in operational costs through automation of routine inquiries.
   - Enhanced data analysis capabilities for better understanding of customer needs and trends.
   - Potential for scaling support operations without proportional increases in staffing.

3) **Risks & constraints:**
   - Initial investment costs and ongoing maintenance expenses may strain budgets.
   - Integration challenges with existing systems and processes.
   - Potential for customer dissatisfaction if AI fails to meet expectations or lacks human touch.
   - Data privacy and security concerns related to customer information handling.

4) **Recommendation:** Conduct a feasibility stud

In [ ]:
state = {
    "query": "How can we build an internal RAG system for HR policies?",
    "intent": "unknown",
    "department": "HR",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=implementation

### A) Summary
The implementation plan outlines a phased approach to develop an internal Retrieval-Augmented Generation (RAG) system for HR policies, leveraging a small team and limited infrastructure. The plan emphasizes simple milestones, security considerations, and manageable dependencies.

### B) Phase Plan

**Phase 0: Discovery & Requirements Gathering**
- Identify key stakeholders in HR and IT.
- Define the scope of HR policies to be included.
- Gather requirements for the RAG system (e.g., user interface, search capabilities).
- Assess existing infrastructure and tools available for implementation.

**Phase 1: Design & Prototyping**
- Create a high-level architecture for the RAG system.
- Develop a prototype focusing on a limited set of HR policies.
- Ensure the prototype includes basic security measures (e.g., access controls).
- Gather feedback from stakeholders on the prototype.

**Phase 2: Development & Testing**
- Build the RAG system based

In [ ]:
state = {
    "query": "AI thing for our company… not sure, what do you suggest?",
    "intent": "unknown",
    "department": "unknown",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=unknown

Final intent: unknown


In [ ]:
def ai_clarifier_node(state):
    clarifier_response = """
Your question about "AI for our company" is a bit too broad to give a useful recommendation.

To suggest something that actually fits your company, I need a bit more context:

1) What are the top 2–3 problems or bottlenecks today? (e.g., support, operations, reporting, sales)
2) How big is your team and what tools do you already use? (e.g., CRM, ticketing, ERP)
3) Are you looking for a quick win (1–3 months) or a bigger transformation?
4) Do you handle sensitive data (HR, finance, health, legal) with compliance requirements?
5) What kind of budget and ownership do you realistically have for an AI initiative?

Reply to these, and I’ll propose 1–2 concrete AI use cases that match your reality — not just generic AI hype 🙂
"""

    return {
        "intent": state["intent"],  # remains 'unknown'
        "response_log": state["response_log"] + [clarifier_response]
    }


In [ ]:
state = {
    "query": "AI thing for our company… not sure, what do you suggest?",
    "intent": "unknown",
    "department": "unknown",
    "response_log": []
}

result = ai_clarifier_node(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))




Your question about "AI for our company" is a bit too broad to give a useful recommendation.

To suggest something that actually fits your company, I need a bit more context:

1) What are the top 2–3 problems or bottlenecks today? (e.g., support, operations, reporting, sales)
2) How big is your team and what tools do you already use? (e.g., CRM, ticketing, ERP)
3) Are you looking for a quick win (1–3 months) or a bigger transformation?
4) Do you handle sensitive data (HR, finance, health, legal) with compliance requirements?
5) What kind of budget and ownership do you realistically have for an AI initiative?

Reply to these, and I’ll propose 1–2 concrete AI use cases that match your reality — not just generic AI hype 🙂


Final intent: unknown


# Task
Integrate the `ai_clarifier_node` into the existing LangGraph graph by updating the routing logic to direct 'unknown' intents to this new node, and then add an edge from `ai_clarifier_node` to `END`.

## Integrate ai_clarifier_node into the graph

### Subtask:
Add the new `ai_clarifier_node` to the `StateGraph` and update the routing logic to direct 'unknown' intents to this node. Then, establish an edge from `ai_clarifier_node` to `END`.


**Reasoning**:
The subtask requires modifying the existing graph to include the new `ai_clarifier_node`, updating the routing logic to direct 'unknown' intents to this node, and establishing an edge from `ai_clarifier_node` to `END`. This involves re-defining the graph structure to incorporate these changes.



In [23]:
graph = StateGraph(AdvisorState)

# Add nodes
graph.add_node("intent_classifier", classify_business_intent)
graph.add_node("ai_strategy", ai_strategy_node)
graph.add_node("ai_implementation", ai_implementation_node)
graph.add_node("ai_clarifier", ai_clarifier_node)

# Entry point
graph.set_entry_point("intent_classifier")

# Conditional routing
graph.add_conditional_edges(
    "intent_classifier",
    route_by_business_intent
)

# End edges
graph.add_edge("ai_strategy", END)
graph.add_edge("ai_implementation", END)
graph.add_edge("ai_clarifier", END)

**Reasoning**:
The next step is to modify the `route_by_business_intent` function as per the instructions, so that it directs 'unknown' intents to the newly added `ai_clarifier_node` instead of `END`.



In [24]:
def route_by_business_intent(state: AdvisorState):
    if state["intent"] == "strategy":
        return "ai_strategy"
    elif state["intent"] == "implementation":
        return "ai_implementation"
    else:
        return "ai_clarifier"

**Reasoning**:
The graph needs to be recompiled to incorporate the changes made to the nodes and routing logic, ensuring the new `ai_clarifier_node` and its routing are active.



In [25]:
advisor = graph.compile()

**Reasoning**:
To verify that the `ai_clarifier_node` and the updated routing logic are correctly integrated, the graph should be tested with an 'unknown' intent query.



In [26]:
state = {
    "query": "AI thing for our company… not sure, what do you suggest?",
    "intent": "unknown",
    "department": "unknown",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=unknown

Final intent: unknown


**Reasoning**:
The previous test indicated that the updated `route_by_business_intent` function was not being used by the graph, likely because the graph was built before the function was redefined. To correct this, the graph construction (adding nodes, setting entry point, and adding conditional edges) must be re-executed after the `route_by_business_intent` function has been updated. This ensures the graph references the correct, modified routing logic.



In [27]:
graph = StateGraph(AdvisorState)

# Add nodes
graph.add_node("intent_classifier", classify_business_intent)
graph.add_node("ai_strategy", ai_strategy_node)
graph.add_node("ai_implementation", ai_implementation_node)
graph.add_node("ai_clarifier", ai_clarifier_node)

# Entry point
graph.set_entry_point("intent_classifier")

# Conditional routing
graph.add_conditional_edges(
    "intent_classifier",
    route_by_business_intent
)

# End edges
graph.add_edge("ai_strategy", END)
graph.add_edge("ai_implementation", END)
graph.add_edge("ai_clarifier", END)

**Reasoning**:
The graph needs to be recompiled after re-initializing it with the updated `route_by_business_intent` function to ensure all changes are active in the `advisor` object.



In [28]:
advisor = graph.compile()

**Reasoning**:
The graph needs to be recompiled after re-initializing it with the updated `route_by_business_intent` function to ensure all changes are active in the `advisor` object.



In [29]:
advisor = graph.compile()

**Reasoning**:
Now that the graph has been re-initialized and recompiled with the correct routing logic, it's crucial to re-test it with an 'unknown' intent query to confirm that the `ai_clarifier_node` is correctly invoked and outputs its clarifying questions.



In [30]:
state = {
    "query": "AI thing for our company… not sure, what do you suggest?",
    "intent": "unknown",
    "department": "unknown",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=unknown


Your question about "AI for our company" is a bit too broad to give a useful recommendation.

To suggest something that actually fits your company, I need a bit more context:

1) What are the top 2–3 problems or bottlenecks today? (e.g., support, operations, reporting, sales)
2) How big is your team and what tools do you already use? (e.g., CRM, ticketing, ERP)
3) Are you looking for a quick win (1–3 months) or a bigger transformation?
4) Do you handle sensitive data (HR, finance, health, legal) with compliance requirements?
5) What kind of budget and ownership do you realistically have for an AI initiative?

Reply to these, and I’ll propose 1–2 concrete AI use cases that match your reality — not just generic AI hype 🙂


Final intent: unknown


## Generate README Content

### Subtask:
Generate general content for a README.md file that describes the AI Advisor LangGraph application.


**Reasoning**:
First, I will define a system prompt that sets the persona of a technical writer for the LLM, as requested by the instructions.



In [31]:
system_prompt = """
You are a technical writer tasked with creating a comprehensive and clear README.md file for a LangGraph application. Your goal is to explain the application's purpose, how it works, and how to use it, in a professional and engaging manner.
"""

**Reasoning**:
Next, I will define the task prompt that specifies the required sections and content for the README.md file, as instructed.



In [32]:
task_prompt = """
Generate a comprehensive README.md for the AI Advisor LangGraph application. The README should include the following sections:

# AI Advisor LangGraph Application

## Introduction
- Briefly explain what the AI Advisor is and its purpose (to route and provide advice on AI-related business queries).

## How It Works
- Describe the core components of the LangGraph application:
  - **Intent Classifier (Router):** Explain its role in categorizing queries into 'strategy', 'implementation', or 'unknown'.
  - **AI Strategy Advisor:** Detail its function in providing high-level, business-focused advice for 'strategy' queries.
  - **AI Implementation Planner:** Explain its role in generating practical implementation plans for 'implementation' queries.
  - **AI Clarifier:** Describe its purpose in asking clarifying questions for 'unknown' or ambiguous queries.
- Mention the state management using `TypedDict` (`AdvisorState`).

## Usage Examples
- Provide a short example for each of the following query types, showing the input query and a brief description of the expected output:
  - A 'strategy' query (e.g., "Should we adopt AI for customer support?")
  - An 'implementation' query (e.g., "How can we build an internal RAG system for HR policies?")
  - An 'unknown' query (e.g., "AI thing for our company… what do you suggest?")

Ensure the language is clear, concise, and professional.
"""


**Reasoning**:
Now that both the system prompt and task prompt are defined, I will invoke the language model with these prompts to generate the README content, as instructed.



In [33]:
readme_content = llm.invoke(system_prompt + task_prompt).content
print(readme_content)

```markdown
# AI Advisor LangGraph Application

## Introduction
The AI Advisor LangGraph application is designed to assist businesses in navigating the complexities of artificial intelligence (AI) by providing tailored advice based on user queries. Its primary purpose is to route inquiries related to AI strategy and implementation, ensuring that users receive relevant and actionable insights to enhance their decision-making processes.

## How It Works
The LangGraph application is built around several core components that work together to classify and respond to user queries effectively:

- **Intent Classifier (Router):** This component is responsible for categorizing incoming queries into three distinct categories: 'strategy', 'implementation', or 'unknown'. By accurately identifying the intent behind a query, the application can direct users to the appropriate advisory resources.

- **AI Strategy Advisor:** Once a query is classified as 'strategy', this component provides high-level, 

## Save README File

### Subtask:
Save the generated content into a file named `README.md` in the current directory.


**Reasoning**:
To save the generated README content, I need to open a file named `README.md` in write mode and write the `readme_content` variable to it, then close the file.



In [34]:
with open("README.md", "w") as f:
    f.write(readme_content)
print("README.md has been saved successfully.")

README.md has been saved successfully.


## Final Task

### Subtask:
Confirm that the `README.md` file has been successfully created and is available in the environment.


## Summary:

### Data Analysis Key Findings

*   The `ai_clarifier_node` was successfully integrated into the LangGraph, with routing logic updated to direct 'unknown' intents to this node.
*   An initial attempt to integrate the node failed because the graph was compiled with an outdated definition of the `route_by_business_intent` function; this was corrected by re-executing the graph construction after the function's redefinition.
*   A comprehensive `README.md` file was generated using an LLM, detailing the AI Advisor LangGraph application's purpose, operational mechanisms (including the Intent Classifier, AI Strategy Advisor, AI Implementation Planner, and AI Clarifier nodes, and `AdvisorState` management), and providing usage examples for 'strategy', 'implementation', and 'unknown' queries.
*   The generated `README.md` content was successfully saved to a file named `README.md` in the current directory.

### Insights or Next Steps

*   To prevent issues like the function redefinition problem encountered during graph construction, ensure that all necessary functions are defined *before* they are referenced in graph building steps, or implement a more robust graph compilation and update mechanism.
*   Further development could involve enhancing the `ai_clarifier_node` with more dynamic or context-aware question generation to improve user experience when handling ambiguous queries.
